In [ ]:
import sys
import os
import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

# Инициализация среды
env = gym.make("CartPole-v1")
n_actions = int(env.action_space.n)
state_dim = env.observation_space.shape[0]

# Создание модели нейронной сети
model = models.Sequential([
    layers.InputLayer(shape=(state_dim,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(n_actions, activation='linear')
])

# Функция выбора действия
def get_action(state, epsilon=0):
    state_tensor = np.expand_dims(state, axis=0).astype(np.float32)
    q_values = model(state_tensor, training=False)[0]

    if np.random.random() < epsilon:
        action = np.random.randint(n_actions)
    else:
        action = int(np.argmax(q_values))

    return action

# Проверка
assert model.output_shape == (None, n_actions)
assert model.layers[-1].activation == keras.activations.linear

s, _ = env.reset()
assert np.shape(get_action(s)) == ()

for eps in [0., 0.1, 0.5, 1.0]:
    state_frequencies = np.bincount([get_action(s, epsilon=eps) for i in range(10000)], minlength=n_actions)
    best_action = state_frequencies.argmax()
    assert abs(state_frequencies[best_action] - 10000 * (1 - eps + eps / n_actions)) < 200
    for other_action in range(n_actions):
        if other_action != best_action:
            assert abs(state_frequencies[other_action] - 10000 * (eps / n_actions)) < 200
    print('e=%.1f tests passed' % eps)

# Компиляция модели
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
loss_fn = tf.keras.losses.MeanSquaredError()
gamma = 0.99

# Функция обучения на батче
@tf.function
def train_on_batch(states, actions, rewards, next_states, dones):
    """
    Обучает модель на батче переходов
    """
    with tf.GradientTape() as tape:
        current_q = model(states, training=True)

        batch_size = tf.shape(states)[0]
        indices = tf.stack([tf.range(batch_size), actions], axis=1)
        current_q_for_actions = tf.gather_nd(current_q, indices)

        next_q = model(next_states, training=False)
        max_next_q = tf.reduce_max(next_q, axis=1)

        targets = rewards + (1 - dones) * gamma * tf.stop_gradient(max_next_q)

        loss = loss_fn(targets, current_q_for_actions)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    return loss

# Функция генерации сессии
def generate_session(epsilon=0, train=False, max_steps=500):
    state, _ = env.reset()
    total_reward = 0

    for step in range(max_steps):
        action = get_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        if train:
            train_on_batch(
                np.array([state], dtype=np.float32),
                np.array([action], dtype=np.int32),
                np.array([reward], dtype=np.float32),
                np.array([next_state], dtype=np.float32),
                np.array([float(done)], dtype=np.float32)
            )

        total_reward += reward
        state = next_state

        if done:
            break

    return total_reward

# Основной цикл обучения
epsilon = 0.5

for epoch in range(500):
    epoch_rewards = [generate_session(epsilon=epsilon, train=True) for _ in range(100)]
    mean_reward = np.mean(epoch_rewards)

    epsilon = max(epsilon * 0.99, 0.01)

    print(f"epoch #{epoch:3d}\tmean reward = {mean_reward:.3f}\tepsilon = {epsilon:.3f}")

    if mean_reward > 300:
        print("You Win!")
        break

env.close()

e=0.0 tests passed
e=0.1 tests passed
e=0.5 tests passed
e=1.0 tests passed
epoch #  0	mean reward = 12.840	epsilon = 0.495
epoch #  1	mean reward = 13.730	epsilon = 0.490
epoch #  2	mean reward = 13.620	epsilon = 0.485
epoch #  3	mean reward = 13.790	epsilon = 0.480
epoch #  4	mean reward = 13.990	epsilon = 0.475
epoch #  5	mean reward = 14.250	epsilon = 0.471
epoch #  6	mean reward = 13.990	epsilon = 0.466
epoch #  7	mean reward = 14.720	epsilon = 0.461
epoch #  8	mean reward = 15.380	epsilon = 0.457
epoch #  9	mean reward = 14.840	epsilon = 0.452
epoch # 10	mean reward = 13.330	epsilon = 0.448
epoch # 11	mean reward = 16.210	epsilon = 0.443
epoch # 12	mean reward = 22.470	epsilon = 0.439
epoch # 13	mean reward = 21.560	epsilon = 0.434
epoch # 14	mean reward = 28.170	epsilon = 0.430
epoch # 15	mean reward = 28.870	epsilon = 0.426
epoch # 16	mean reward = 38.510	epsilon = 0.421
epoch # 17	mean reward = 44.130	epsilon = 0.417
epoch # 18	mean reward = 45.590	epsilon = 0.413
epoch # 19	m